# Построчное преобразование XLSB в CSV

Поместите блокнот рядом с папкой `download_reestr` и запустите ячейки по порядку. Результаты сохраняются в `download_reestr_csv_misha` с сохранением структуры папок.

Читается первый лист. Все строки, включая заголовки, записываются как данные без добавления служебного заголовка. Разделитель — запятая, кодировка — UTF-8 с BOM. Форматирование Excel не переносится; даты могут записываться как числовые значения Excel.

В основной ячейке можно изменить START_ROW (первая сохраняемая строка), COLUMN_COUNT (число столбцов) и OVERWRITE (перезаписывать готовые CSV). По умолчанию сохраняются все строки и столбцы. Для продолжения после остановки задайте OVERWRITE = False. Готовый CSV заменяется только после успешной записи всего файла.


In [ ]:
%pip install pyxlsb tqdm


In [ ]:
from pathlib import Path
import csv
from time import perf_counter
from pyxlsb import open_workbook
from tqdm import tqdm

source_dir = Path("download_reestr")
target_dir = Path("download_reestr_csv_misha")

# Номера строк и листов начинаются с 1.
SHEET_NUMBER = 1
START_ROW = 1
COLUMN_COUNT = None  # Можно указать число столбцов, например 25.
OVERWRITE = True

if not source_dir.is_dir():
    raise FileNotFoundError(f"Не найдена папка: {source_dir.resolve()}")
if START_ROW < 1 or (COLUMN_COUNT is not None and COLUMN_COUNT < 1):
    raise ValueError("START_ROW и COLUMN_COUNT должны быть положительными.")

files = sorted(
    p for p in source_dir.rglob("*")
    if p.is_file() and p.suffix.lower() == ".xlsb"
    and not p.name.startswith("~$")
)
print(f"Найдено файлов: {len(files)}")
converted = 0
skipped = 0
errors = []
started = perf_counter()

with tqdm(total=len(files), desc="Обработано файлов", unit="файл") as progress:
    for number, source_file in enumerate(files, start=1):
        relative_path = source_file.relative_to(source_dir)
        target_file = (target_dir / relative_path).with_suffix(".csv")
        temporary_file = target_file.with_name(target_file.name + ".part")
        file_started = perf_counter()
        row_count = 0
        try:
            if target_file.exists() and not OVERWRITE:
                skipped += 1
                tqdm.write(f"[{number}/{len(files)}] Уже существует: {relative_path}")
                continue

            target_file.parent.mkdir(parents=True, exist_ok=True)
            with open_workbook(str(source_file)) as workbook:
                with workbook.get_sheet(SHEET_NUMBER) as sheet:
                    with temporary_file.open("w", newline="", encoding="utf-8-sig") as output:
                        writer = csv.writer(output, delimiter=",")
                        next_row = START_ROW - 1
                        for row in sheet.rows():
                            if not row:
                                continue
                            row_index = row[0].r
                            if row_index < START_ROW - 1:
                                continue
                            # Сохраняем пропущенные пустые строки и позиции столбцов.
                            while next_row < row_index:
                                writer.writerow([""] * (COLUMN_COUNT or 0))
                                row_count += 1
                                next_row += 1
                            width = COLUMN_COUNT if COLUMN_COUNT is not None else max(cell.c for cell in row) + 1
                            values = [""] * width
                            for cell in row:
                                if cell.c < width:
                                    values[cell.c] = "" if cell.v is None else cell.v
                            writer.writerow(values)
                            row_count += 1
                            next_row = row_index + 1

            temporary_file.replace(target_file)
            converted += 1
            elapsed = perf_counter() - file_started
            tqdm.write(
                f"[{number}/{len(files)}] Готово: {relative_path} | "
                f"строк: {row_count:,} | {elapsed:.1f} с"
            )
        except Exception as error:
            temporary_file.unlink(missing_ok=True)
            errors.append((str(relative_path), str(error)))
            tqdm.write(f"[{number}/{len(files)}] Ошибка: {relative_path} — {error}")
        finally:
            progress.set_postfix(готово=converted, ошибок=len(errors), refresh=False)
            progress.update(1)

print(f"\nПреобразовано: {converted} из {len(files)}")
print(f"Пропущено существующих: {skipped}. Ошибок: {len(errors)}")
print(f"Общее время: {(perf_counter() - started) / 60:.1f} мин")
print(f"Результат: {target_dir.resolve()}")
